### CONFIGURACIÓN Y VALIDACIÓN DE LA CONEXIÓN A LA BD CON PostGIS

In [10]:
import psycopg2

def probar_conexion_real():
    """
    Módulo: Adquisición / Tubería Inicial
    Filtro de Validación: Conexión local directa mediante el venv.
    """
    
    DB_PARAMS = {
        "host": "localhost",
        "database": "Tesis_geoespacial", 
        "user": "postgres",                       
        "password": "43959421",
        "port": "5432"    
    }
    
    try:
        print("[SIGMA-ML] Conectando localmente a la base de datos...")
        conn = psycopg2.connect(**DB_PARAMS)
        cur = conn.cursor()
        
        # Verifica que PostGIS está activo en la BD
        cur.execute("SELECT PostGIS_Version();")
        version = cur.fetchone()
        
        print(f"\n Conexión Exitosa con el entorno (venv)!")
        print(f"-> PostGIS activo en la base de datos: {version[0]}")
        
        cur.close()
        conn.close()
        return DB_PARAMS
        
    except Exception as e:
        print(f"\n Error al conectar: {e}")
        print("Revisa que el nombre de tu BD, usuario o contraseña estén bien escritos y que pgAdmin esté abierto.")

# Ejecutar la función de prueba
config_bd = probar_conexion_real()

[SIGMA-ML] Conectando localmente a la base de datos...

 Conexión Exitosa con el entorno (venv)!
-> PostGIS activo en la base de datos: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


### COMPONENTE 1: ADQUISICIÓN Y PREPARACIÓN DE DATOS GEOESPACIALES

In [11]:
# INICIO DEL PIPELINE Y SEPARACIÓN DEL ID

from datetime import datetime

def registrar_inicio_pipeline(params_bd):
    """
    Módulo 1: Adquisición / Registro Inicial
    Inserta el estado inicial basándose en el modelo físico.
    """
    # Consulta SQL adaptada a los campos de la tabla procesamiento_automatico
    sql_insert = """
    INSERT INTO procesamiento_automatico (fecha_hora, tiempo_ejecucion, umbral_longitud, estado, tipo_procesamiento)
    VALUES (%s, %s, %s, %s, %s)
    RETURNING id_procesamiento;
    """
    
    id_generado = None
    
    try:
        # Conectamos usando los parámetros validados
        conn = psycopg2.connect(**params_bd)
        cur = conn.cursor()
        
        # Valores que coinciden con los tipos de datos de tu tabla
        fecha_actual = datetime.now()             # Para 'fecha_hora' (timestamp)
        estado_inicial = "EN PROCESO"             # Para 'estado' (varchar 25)
        tipo_proc = "CLASIFICACION"               # Para 'tipo_procesamiento' (varchar 20)
        tiempo_inicial = 0                        # Enviamos 0 temporalmente para cumplir la restricción NOT NULL
        umbral_inicial = 0                        # Enviamos 0 temporalmente para cumplir la restricción NOT NULL
        
        # Ejecutar la inserción en la base de datos
        cur.execute(sql_insert, (fecha_actual, tiempo_inicial, umbral_inicial, estado_inicial, tipo_proc))
        
        # Recuperar el id_procesamiento autogenerado (serial)
        id_generado = cur.fetchone()[0]
        
        # Confirmar la transacción para que se guarde físicamente
        conn.commit()
        
        print(f" Módulo 1 Completado con éxito!")
        print(f"-> Registro insertado en la tabla 'procesamiento_automatico'")
        print(f"-> ID de Ejecución Generado: {id_generado}")
        
        cur.close()
        conn.close()
        return id_generado
        
    except Exception as e:
        print(f" Error al insertar en la tabla: {e}")
        if 'conn' in locals():
            conn.rollback()  # Deshace cambios si hubo error para no congelar la BD
        return None

# Ejecutamos el inicio del proceso usando la configuración de la primera celda
if 'config_bd' in locals() and config_bd:
    id_proceso = registrar_inicio_pipeline(config_bd)
else:
    print(" Error: No se encontró la configuración de la BD. Ejecuta la celda de arriba primero.")

 Módulo 1 Completado con éxito!
-> Registro insertado en la tabla 'procesamiento_automatico'
-> ID de Ejecución Generado: 1


In [4]:
# SIMULACIÓN DE CARGA DE IMÁGENES (SATELITAL Y DEM)

def simular_carga_insumos_geoespaciales(params_bd):
    """
    Inserta los registros de prueba en la tabla 'insumo_geoespacial'
    configurando el pipeline para el nuevo estándar de seguridad de 20m.
    """
    sql_insert = """
    INSERT INTO insumo_geoespacial (ruta_almacenamiento, fuente, precision_horizontal, resolucion_espacial, epsg, x_min, x_max, y_min, y_max, tipo_insumo)
    VALUES 
    ('/data/mitu_sentinel2_10m.tif', 'Sentinel-2', 10.0, 10.0, 32718, 550000, 600000, 8400000, 8450000, 'SATELITAL'),
    ('/data/mitu_alospalsar_resample_10m.tif', 'ALOS PALSAR', 12.5, 10.0, 32718, 550000, 600000, 8400000, 8450000, 'DEM');
    """
    try:
        conn = psycopg2.connect(**params_bd)
        cur = conn.cursor()
        cur.execute(sql_insert)
        conn.commit()
        print("REGISTROS INSERTADOS: El sistema operará con el umbral óptimo (>20m).")
        cur.close()
        conn.close()
    except Exception as e:
        print(f"Error al insertar simulación: {e}")

simular_carga_insumos_geoespaciales(config_bd)

REGISTROS INSERTADOS: El sistema operará con el umbral óptimo (>20m).


In [12]:
# FILTRO 1: ADQUISICIÓN DE INSUMOS GEOESPACIALES (IMAGEN SATELITAL + DEM)

def filtro1_adquisicion_insumos(params_bd, id_ejecucion):
    """
    COMPONENTE: ADQUISICIÓN Y PREPARACIÓN DE DATOS GEOESPACIALES
    Filtro 1: Adquisición de Datos
    Objetivo: Adquirir de forma secuencial AMBOS insumos (Imagen Satelital y DEM)
              utilizando las columnas reales del modelo físico.
    """
    sql_select = """
    SELECT id_insumo, ruta_almacenamiento, tipo_insumo, fuente, resolucion_espacial
    FROM insumo_geoespacial
    WHERE UPPER(tipo_insumo) IN ('SATELITAL', 'DEM', 'IMAGEN SATELITAL', 'IMAGEN DEM')
    LIMIT 10;
    """
    
    try:
        conn = psycopg2.connect(**params_bd)
        cur = conn.cursor()
        
        print(f"[SIGMA-ML] [Filtro 1: Adquisición] Iniciando descarga secuencial de insumos desde 'insumo_geoespacial'...")
        cur.execute(sql_select)
        insumos_registrados = cur.fetchall()
        
        lista_satelital = []
        lista_dem = []
        
        for insumo in insumos_registrados:
            tipo = str(insumo[2]).upper()
            if "SAT" in tipo:
                lista_satelital.append(insumo)
            elif "DEM" in tipo:
                lista_dem.append(insumo)
        
        print(f"\nFiltro 1 (Adquisición) Ejecutado:")
        
        resultado = None
        if lista_satelital and lista_dem:
            print(f"-> [OK] Imagen Satelital Detectada: {lista_satelital[0][1]} (Fuente: {lista_satelital[0][3]})")
            print(f"-> [OK] Modelo Digital de Elevación (DEM) Detectado: {lista_dem[0][1]} (Fuente: {lista_dem[0][3]})")
            print(f"\n[INFO] El sistema ha acoplado exitosamente ambos componentes de percepción remota.")
            resultado = {"id_satelite": lista_satelital[0][0], "id_dem": lista_dem[0][0]}
        else:
            print("Alerta: El pipeline requiere obligatoriamente de una Imagen Satelital y un DEM en la tabla.")
            print(f"Insumos Satelitales encontrados: {len(lista_satelital)} | Insumos DEM encontrados: {len(lista_dem)}")
        
        cur.close()
        conn.close()
        return resultado
        
    except Exception as e:
        print(f"Error en la Adquisición de Datos: {e}")
        return None

# =======================================
# CONTROL INTEGRADO CON LA BASE DE DATOS
# =======================================

# 1. Intentamos recuperar el ID desde la memoria de VS Code
id_final = locals().get('id_procesamiento') or locals().get('id_proceso')

# 2. Si se borró de memoria por el reinicio del Kernel, consultamos el último ID real en pgAdmin
if not id_final:
    try:
        conn = psycopg2.connect(**config_bd)
        cur = conn.cursor()
        cur.execute("SELECT MAX(id_procesamiento) FROM public.procesamiento_automatico;")
        id_final = cur.fetchone()[0]
        cur.close()
        conn.close()
    except Exception:
        id_final = None

# 3. Ejecución segura
if id_final:
    id_procesamiento = id_final  # Seteamos el nombre definitivo
    insumos_cargados = filtro1_adquisicion_insumos(config_bd, id_procesamiento)
else:
    print("Error: No se encontró ningún ID de control activo en la base de datos.")

[SIGMA-ML] [Filtro 1: Adquisición] Iniciando descarga secuencial de insumos desde 'insumo_geoespacial'...

Filtro 1 (Adquisición) Ejecutado:
-> [OK] Imagen Satelital Detectada: /data/mitu_sentinel2_10m.tif (Fuente: Sentinel-2)
-> [OK] Modelo Digital de Elevación (DEM) Detectado: /data/mitu_alospalsar_resample_10m.tif (Fuente: ALOS PALSAR)

[INFO] El sistema ha acoplado exitosamente ambos componentes de percepción remota.
